In [1]:
import os
import glob

import numpy as np
import torch
import torch.nn as nn
import time

from torch.utils.data import Dataset, DataLoader, Sampler

from tqdm.auto import tqdm

c:\Users\louiz\Documents\Projects\phoneme-lipsync\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
SAMPLE_RATE = 16000

N_MELS = 80
TIME_FRAMES = 49
NUM_CLASSES = 41

BATCH_SIZE = 32

LEARNING_RATE = 1e-3
EPOCHS = 10

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

TRAIN_DIR = "../data/processed/phoneme_chunks/train"
VAL_DIR = "../data/processed/phoneme_chunks/validation"
TEST_DIR = "../data/processed/phoneme_chunks/test"

CHECKPOINT_DIR = "../checkpoints"

os.makedirs(
    CHECKPOINT_DIR,
    exist_ok=True
)

print("Device:", DEVICE)

if torch.cuda.is_available():
    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

Device: cuda
GPU: NVIDIA GeForce RTX 3050 Laptop GPU


In [3]:
class PhonemeShardDataset(Dataset):

    def __init__(self, shard_dir):

        self.shard_dir = shard_dir

        self.shard_files = sorted(
            glob.glob(
                os.path.join(
                    shard_dir,
                    "shard_*.npz"
                )
            )
        )

        self.shard_sizes = []

        for path in self.shard_files:
            with np.load(path) as shard:
                self.shard_sizes.append(
                    shard["X"].shape[0]
                )

        self.cumulative_sizes = np.cumsum(
            self.shard_sizes
        )

        self.total_size = int(
            self.cumulative_sizes[-1]
        )

        self.cached_shard_id = None
        self.cached_X = None
        self.cached_y = None

    def __len__(self):
        return self.total_size

    def _load_shard(self, shard_id):

        if shard_id == self.cached_shard_id:
            return

        shard = np.load(
            self.shard_files[shard_id]
        )

        self.cached_X = shard["X"]
        self.cached_y = shard["y"]

        shard.close()

        self.cached_shard_id = shard_id

    def __getitem__(self, index):

        if index < 0:
            index += self.total_size

        if index < 0 or index >= self.total_size:
            raise IndexError(
                "Dataset index out of range"
            )

        shard_id = np.searchsorted(
            self.cumulative_sizes,
            index,
            side="right"
        )

        if shard_id == 0:
            local_index = index
        else:
            local_index = (
                index
                - self.cumulative_sizes[
                    shard_id - 1
                ]
            )

        self._load_shard(shard_id)

        X = self.cached_X[local_index]
        y = self.cached_y[local_index]

        return (
            torch.from_numpy(X).float(),
            torch.from_numpy(y).long()
        )

In [4]:
train_dataset = PhonemeShardDataset(
    TRAIN_DIR
)

val_dataset = PhonemeShardDataset(
    VAL_DIR
)

test_dataset = PhonemeShardDataset(
    TEST_DIR
)

print("Training examples:", len(train_dataset))
print("Validation examples:", len(val_dataset))
print("Test examples:", len(test_dataset))

Training examples: 271927
Validation examples: 29024
Test examples: 75160


In [5]:
class ShardBatchSampler(Sampler):

    def __init__(
        self,
        dataset,
        batch_size,
        shuffle=True,
        drop_last=False,
    ):

        self.dataset = dataset
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.drop_last = drop_last

        self.shard_ranges = []

        start = 0

        for shard_size in dataset.shard_sizes:

            end = start + shard_size

            self.shard_ranges.append(
                (start, end)
            )

            start = end

    def __iter__(self):

        shard_order = list(
            range(
                len(self.shard_ranges)
            )
        )

        if self.shuffle:
            np.random.shuffle(
                shard_order
            )

        for shard_id in shard_order:

            start, end = (
                self.shard_ranges[shard_id]
            )

            indices = np.arange(
                start,
                end
            )

            if self.shuffle:
                np.random.shuffle(
                    indices
                )

            for batch_start in range(
                0,
                len(indices),
                self.batch_size
            ):

                batch = indices[
                    batch_start:
                    batch_start + self.batch_size
                ]

                if (
                    len(batch)
                    < self.batch_size
                    and self.drop_last
                ):
                    continue

                yield batch.tolist()

    def __len__(self):

        total_batches = 0

        for shard_size in self.dataset.shard_sizes:

            if self.drop_last:
                total_batches += (
                    shard_size
                    // self.batch_size
                )
            else:
                total_batches += int(
                    np.ceil(
                        shard_size
                        / self.batch_size
                    )
                )

        return total_batches

In [6]:
train_batch_sampler = ShardBatchSampler(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=False,
)

val_batch_sampler = ShardBatchSampler(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    drop_last=False,
)

test_batch_sampler = ShardBatchSampler(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    drop_last=False,
)

train_loader = DataLoader(
    train_dataset,
    batch_sampler=train_batch_sampler,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)

val_loader = DataLoader(
    val_dataset,
    batch_sampler=val_batch_sampler,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)

test_loader = DataLoader(
    test_dataset,
    batch_sampler=test_batch_sampler,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)

print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))

Training batches: 8566
Validation batches: 914
Test batches: 2368


In [7]:
X_batch, y_batch = next(
    iter(train_loader)
)

print("X:", X_batch.shape)
print("y:", y_batch.shape)

X: torch.Size([32, 80, 49])
y: torch.Size([32, 49])


In [8]:
class CNNGRU(nn.Module):

    def __init__(
        self,
        num_classes,
        hidden_size=192,
    ):
        super().__init__()

        self.cnn = nn.Sequential(

            nn.Conv2d(
                1,
                32,
                kernel_size=3,
                padding=1,
            ),

            nn.BatchNorm2d(32),

            nn.ReLU(),

            nn.MaxPool2d(
                kernel_size=(2, 1)
            ),

            nn.Conv2d(
                32,
                64,
                kernel_size=3,
                padding=1,
            ),

            nn.BatchNorm2d(64),

            nn.ReLU(),

            nn.MaxPool2d(
                kernel_size=(2, 1)
            ),
        )

        cnn_feature_size = 64 * 20

        self.gru = nn.GRU(
            input_size=cnn_feature_size,
            hidden_size=hidden_size,
            num_layers=1,
            batch_first=True,
        )

        self.classifier = nn.Linear(
            hidden_size,
            num_classes,
        )

    def forward(self, x):

        x = x.unsqueeze(1)

        x = self.cnn(x)

        x = x.permute(
            0, 3, 1, 2
        )

        x = x.flatten(
            start_dim=2
        )

        x, _ = self.gru(x)

        x = self.classifier(x)

        return x

In [9]:
model = CNNGRU(
    num_classes=NUM_CLASSES,
    hidden_size=192,
).to(DEVICE)

criterion = nn.CrossEntropyLoss(
    ignore_index=-100
)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE
)

print(model)

CNNGRU(
  (cnn): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=(2, 1), stride=(2, 1), padding=0, dilation=1, ceil_mode=False)
    (4): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (5): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (6): ReLU()
    (7): MaxPool2d(kernel_size=(2, 1), stride=(2, 1), padding=0, dilation=1, ceil_mode=False)
  )
  (gru): GRU(1280, 192, batch_first=True)
  (classifier): Linear(in_features=192, out_features=41, bias=True)
)


In [10]:
X_batch = X_batch.to(
    DEVICE,
    non_blocking=True
)

y_batch = y_batch.to(
    DEVICE,
    non_blocking=True
)

output = model(X_batch)

loss = criterion(
    output.reshape(-1, NUM_CLASSES),
    y_batch.reshape(-1)
)

print("Input:", X_batch.shape)
print("Output:", output.shape)
print("Loss:", loss.item())

Input: torch.Size([32, 80, 49])
Output: torch.Size([32, 49, 41])
Loss: 3.751920461654663


In [11]:
def evaluate(model, loader):

    model.eval()

    total_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():

        for X, y in loader:

            X = X.to(
                DEVICE,
                non_blocking=True
            )

            y = y.to(
                DEVICE,
                non_blocking=True
            )

            output = model(X)

            loss = criterion(
                output.reshape(
                    -1,
                    NUM_CLASSES
                ),
                y.reshape(-1)
            )

            total_loss += (
                loss.item()
                * y.numel()
            )

            predictions = output.argmax(
                dim=-1
            )

            valid = y != -100

            correct += (
                (
                    predictions[valid]
                    == y[valid]
                )
                .sum()
                .item()
            )

            total += valid.sum().item()

    average_loss = (
        total_loss / total
    )

    accuracy = (
        correct / total
    )

    return average_loss, accuracy

In [12]:
# TRAINING

best_val_loss = float("inf")

history = []

for epoch in range(1, EPOCHS + 1):

    model.train()

    total_loss = 0.0
    total_frames = 0

    start_time = time.time()

    progress_bar = tqdm(
        train_loader,
        desc=f"Epoch {epoch}/{EPOCHS}",
        unit="batch"
    )

    for batch_index, (X, y) in enumerate(
        progress_bar,
        start=1
    ):

        X = X.to(
            DEVICE,
            non_blocking=True
        )

        y = y.to(
            DEVICE,
            non_blocking=True
        )

        optimizer.zero_grad()

        output = model(X)

        loss = criterion(
            output.reshape(
                -1,
                NUM_CLASSES
            ),
            y.reshape(-1)
        )

        loss.backward()

        optimizer.step()

        valid_frames = (
            y != -100
        ).sum().item()

        total_loss += (
            loss.item()
            * valid_frames
        )

        total_frames += valid_frames

        running_loss = (
            total_loss
            / total_frames
        )

        progress_bar.set_postfix(
            loss=f"{running_loss:.4f}"
        )

    train_loss = (
        total_loss
        / total_frames
    )

    # Validation
    val_loss, val_accuracy = evaluate(
        model,
        val_loader
    )

    epoch_time = (
        time.time()
        - start_time
    )

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "val_accuracy": val_accuracy,
        "time": epoch_time,
    })

    print(
        f"\nEpoch {epoch}/{EPOCHS} complete"
    )

    print(
        f"Train loss: {train_loss:.4f}"
    )

    print(
        f"Val loss:   {val_loss:.4f}"
    )

    print(
        f"Val accuracy: {val_accuracy:.2%}"
    )

    print(
        f"Time: {epoch_time / 60:.2f} min"
    )

    if val_loss < best_val_loss:

        best_val_loss = val_loss

        torch.save(
            model.state_dict(),
            os.path.join(
                CHECKPOINT_DIR,
                "best_cnn_gru_192.pt"
            )
        )

        print(
            "✓ New best model saved."
        )

Epoch 1/10: 100%|██████████| 8566/8566 [03:35<00:00, 39.68batch/s, loss=0.9847]



Epoch 1/10 complete
Train loss: 0.9847
Val loss:   1.1467
Val accuracy: 65.88%
Time: 3.81 min
✓ New best model saved.


Epoch 2/10: 100%|██████████| 8566/8566 [03:35<00:00, 39.77batch/s, loss=0.8059]



Epoch 2/10 complete
Train loss: 0.8059
Val loss:   1.0729
Val accuracy: 68.18%
Time: 3.80 min
✓ New best model saved.


Epoch 3/10: 100%|██████████| 8566/8566 [03:34<00:00, 39.95batch/s, loss=0.7574]


...


Epoch 8/10 complete
Train loss: 0.6756
Val loss:   1.0008
Val accuracy: 70.26%
Time: 3.81 min


Epoch 9/10: 100%|██████████| 8566/8566 [03:36<00:00, 39.58batch/s, loss=0.6690]



Epoch 9/10 complete
Train loss: 0.6690
Val loss:   1.0058
Val accuracy: 70.25%
Time: 3.82 min


Epoch 10/10: 100%|██████████| 8566/8566 [03:36<00:00, 39.50batch/s, loss=0.6610]



Epoch 10/10 complete
Train loss: 0.6610
Val loss:   0.9939
Val accuracy: 70.66%
Time: 3.83 min
